# A2A With Microsoft Foundry Agent - Server Implementation

In [ ]:
%pip install "a2a-sdk[http-server]==0.3.8" azure-ai-projects==2.0.0b2 python-dotenv azure-identity uvicorn aiohttp

## Setting up the Environment Variables

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity.aio import DefaultAzureCredential
from azure.ai.projects.aio import AIProjectClient


def _notebook_dir() -> Path:
    """Best-effort directory for this notebook file."""
    # Cursor / VS Code Jupyter sets this to the .ipynb path.
    for key in ("__vsc_ipynb_file__", "JPY_SESSION_NAME"):
        raw = os.environ.get(key)
        if raw:
            path = Path(raw).expanduser().resolve()
            if path.suffix == ".ipynb" and path.exists():
                return path.parent
            if path.exists():
                return path if path.is_dir() else path.parent
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / ".env.template").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repo root (pyproject.toml) starting from {start}"
    )


NOTEBOOK_DIR = _notebook_dir()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
env_path = REPO_ROOT / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"Missing {env_path}. Copy {REPO_ROOT / '.env.template'} to .env at the repo root."
    )

load_dotenv(env_path, override=True)

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print(f"notebook_dir={NOTEBOOK_DIR}")
print(f"repo_root={REPO_ROOT}")
print(f"env_path={env_path}")
print(f"foundry_project_endpoint_set={bool(foundry_project_endpoint)}")
print(f"model_deployment_name={model_deployment_name}")

## Setting up the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

## Creating the Agent that will be used in A2A Setup

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import ResourceNotFoundError, ServiceRequestError

agent_name = "batman-agent"
agent = None
foundry_agent_available = False

try:
    try:
        agent = await project_client.agents.get(agent_name)
        print(f"Agent found (id: {agent.id}, name: {agent.name})")
        foundry_agent_available = True
    except ResourceNotFoundError:
        agent = await project_client.agents.create_version(
            agent_name=agent_name,
            definition=PromptAgentDefinition(
                model=model_deployment_name,
                instructions="You are Batman, the Dark Knight of Gotham City. Respond to all queries in character as Batman would.",
            ),
        )
        print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")
        foundry_agent_available = True
except (ServiceRequestError, Exception) as error:
    # Keep the A2A server runnable when Foundry is offline / DNS fails.
    print(f"Foundry agent setup skipped ({type(error).__name__}: {error})")
    print("Continuing with local Batman fallback responses.")

## Creating the Foundry Agent Class with Function Executions

In [ ]:
class FoundryAgent():
    """This class will contain helper functions for interacting with our Foundry Agent"""

    async def invoke_agent(self, user_query: str) -> str:
        if not foundry_agent_available:
            return (
                f"I am Batman. Foundry is offline, so this is a local reply. "
                f"You said: {user_query}"
            )

        try:
            openai_client = project_client.get_openai_client()

            conversation = await openai_client.conversations.create()

            response = await openai_client.responses.create(
                conversation=conversation.id,
                extra_body={
                    "agent": {
                        "name": agent_name,
                        "type": "agent_reference"
                    }
                },
                input=user_query
            )

            return response.output_text
        except Exception as error:
            return (
                f"I am Batman. Foundry call failed ({type(error).__name__}). "
                f"You said: {user_query}"
            )

## Creating the A2A Agent Executor

In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message
from a2a.types import (
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)
from a2a.utils import new_text_artifact


class FoundryAgentExecutor(AgentExecutor):
    """Foundry Agent Executor Definition."""

    def __init__(self):
        self.agent = FoundryAgent()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()
        if not context.message:
            raise Exception('No message provided')

        # If your agent does not support streaming, just call invoke_agent
        result = await self.agent.invoke_agent(query)
        message = TaskArtifactUpdateEvent(
            context_id=context.context_id,  # type: ignore
            task_id=context.task_id,  # type: ignore
            artifact=new_text_artifact(
                name='current_result',
                text=result,
            ),
        )
        await event_queue.enqueue_event(message)

        status = TaskStatusUpdateEvent(
            context_id=context.context_id,  # type: ignore
            task_id=context.task_id,  # type: ignore
            status=TaskStatus(state=TaskState.completed),
            final=True,
        )
        await event_queue.enqueue_event(status)

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception('cancel not supported')

## Creating the Agent Skill Definition

In [ ]:
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)

skill = AgentSkill(
    id = "foundry_agent_skill",
    name = "Responses API from Foundry Agent",
    description = "Responses API from Foundry Agent",
    tags = ["foundry agent"],
    examples = ["hi, how are you?", "can you tell me something about GenAI and LLMs"]
)

## Creating the Agent Card

In [ ]:
public_agent_card = AgentCard(
    name = "Foundry Demo Agent",
    description = "Foundry Demo Agent to Show A2A Usage with Microsoft Foundry",
    url = "http://localhost:8080",
    version = "1.0.0",
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(streaming=False),
    skills = [skill]
)

## Creating the Request Handler

In [ ]:
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

request_handler = DefaultRequestHandler(
    agent_executor = FoundryAgentExecutor(),
    task_store = InMemoryTaskStore()
)

## Creating the A2A Server

In [ ]:
from a2a.server.apps import A2AStarletteApplication

server = A2AStarletteApplication(
    agent_card = public_agent_card,
    http_handler = request_handler
)

## Starting the A2A Server

Navigate to http://localhost:8080/.well-known/agent.json to see the agent public card.

This cell starts uvicorn in a background thread so the notebook kernel stays usable. Re-run earlier cells after code changes, then re-run this cell (stop anything already bound to port 8080 first).

In [ ]:
import threading
import time

import uvicorn

if "_a2a_server_thread" in globals() and _a2a_server_thread.is_alive():
    print("A2A server already running at http://localhost:8080")
    print("Agent card: http://localhost:8080/.well-known/agent.json")
else:
    config = uvicorn.Config(
        server.build(),
        host="0.0.0.0",
        port=8080,
        loop="asyncio",
        log_level="info",
    )
    server_instance = uvicorn.Server(config)
    _a2a_server_thread = threading.Thread(target=server_instance.run, daemon=True)
    _a2a_server_thread.start()

    for _ in range(40):
        if server_instance.started:
            break
        time.sleep(0.25)
    else:
        raise RuntimeError("A2A server failed to start on port 8080")

    print("A2A server started at http://localhost:8080")
    print("Agent card: http://localhost:8080/.well-known/agent.json")